# CPC353 Assignment 1 — N-gram Analysis

This notebook analyses bigram and trigram patterns in the combined instant noodle review corpus. The 2026-11-09 snapshot contains all 125 review rows (57 collected, 68 AI-generated) across products; no contributor, ID-prefix or language filter is applied.

Run `preprocessing.ipynb` first. The notebook saves analysis tables in `data/analysis/ngram/`.

**Research context:** N-gram analysis helps identify frequently co-occurring word pairs and triples that reveal product-attribute discussions and code-switching patterns. Cross-language bigrams (English–Malay adjacencies) are particularly informative for understanding how reviewers switch between languages within phrases.

## 1. Imports and locate the project folder

The notebook searches the current folder and its parents for the preprocessing outputs, so it works when launched from the repository root or a subfolder.

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

PROJECT_DIR = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "processed").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find data/processed. Run this notebook from the project.")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "data" / "analysis" / "ngram"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

print(f"Project folder: {PROJECT_DIR}")
print(f"N-gram outputs: {OUTPUT_DIR}")

Project folder: C:\Users\Admin\OneDrive\Documents\GitHub\CPC353-Assignment-1
N-gram outputs: C:\Users\Admin\OneDrive\Documents\GitHub\CPC353-Assignment-1\data\analysis\ngram


## 2. Load the retained reviews and project stop words

All included collected and AI-generated reviews are combined into one corpus. We load the project stop-word list to produce both unfiltered and filtered n-gram views.

In [2]:
def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        rows = [json.loads(line) for line in file if line.strip()]
    return [row for row in rows if row.get("included", True)]

reviews = (
    load_jsonl(PROCESSED_DIR / "real_preprocessed.jsonl")
    + load_jsonl(PROCESSED_DIR / "ai_preprocessed.jsonl")
)

stopwords_df = pd.read_csv(PROCESSED_DIR / "stopwords_used.csv")
stopwords_df["word"] = stopwords_df["word"].astype(str).str.casefold()
stopword_set = set(stopwords_df["word"])

print(f"Included reviews in combined corpus: {len(reviews)}")
print(f"Project stop words loaded: {len(stopword_set)}")

Included reviews in combined corpus: 125
Project stop words loaded: 100


## 3. Build a bilingual word-language lookup

To identify cross-language n-grams, we build a simple word-to-language lookup from the project stop-word list, protected words, and the concordance lexicon. This uses the project's own classifications, not automatic language detection. Words not in the lookup are labelled by a simple heuristic based on common Malay word patterns; ambiguous cases are marked `ambiguous`.

In [3]:
import csv

# Load the stop-word language labels
word_lang = {}
for _, row in stopwords_df.iterrows():
    word_lang[row["word"]] = row["language"]

# Load protected words
protected_df = pd.read_csv(PROCESSED_DIR / "protected_words.csv")
for _, row in protected_df.iterrows():
    w = str(row["word"]).casefold()
    if w not in word_lang and "reason" in row.index:
        reason = str(row["reason"]).lower()
        if "malay" in reason or "bahasa" in reason:
            word_lang[w] = "Malay"
        elif "english" in reason:
            word_lang[w] = "English"

# Load the concordance lexicon
lexicon_path = PROJECT_DIR / "config" / "concordance_lexicon.csv"
if lexicon_path.exists():
    with lexicon_path.open(encoding="utf-8-sig") as f:
        for entry in csv.DictReader(f):
            expr = entry["expression"].casefold().strip()
            lang = entry.get("language", "").strip()
            if " " not in expr:  # single words only for token lookup
                if "malay" in lang.lower():
                    word_lang[expr] = "Malay"
                elif "english" in lang.lower():
                    word_lang[expr] = "English"

# Common Malay content words from the corpus (manually verified)
KNOWN_MALAY = {
    "sedap", "pedas", "kuah", "kuahnya", "kenyal", "rasa", "mi", "kari",
    "lembik", "masin", "mahal", "harga", "bungkusan", "masak", "rebus",
    "goreng", "makan", "suka", "sangat", "memang", "jugak", "perlu",
    "sesuai", "tekak", "makanan", "berbeza", "buat", "celur", "tambah",
    "masih", "agak", "cair", "pekat", "rangup", "lemak", "panas",
    "enak", "nikmat", "lazat", "sedikit", "banyak", "topping",
    "gila", "punya", "bila", "boleh", "ganti", "siapa", "pandai",
    "tengok", "letak", "pon", "mcm", "jela", "mmg", "sgt",
    "kaw", "firma", "kalau", "jadi", "dekat", "sebab", "lagi",
}
KNOWN_ENGLISH = {
    "noodles", "noodle", "instant", "spicy", "taste", "flavour", "flavor",
    "broth", "soup", "texture", "springy", "chewy", "salty", "crispy",
    "price", "pricey", "packaging", "packet", "portion", "good", "best",
    "love", "like", "enjoy", "recommend", "worth", "cooking", "cook",
    "rich", "bold", "satisfying", "delicious", "amazing", "perfect",
    "firm", "soft", "soggy", "bland", "thick", "thin", "hot",
    "spice", "heat", "kick", "seasoning", "condiment", "flakes",
    "favourite", "favorite", "definitely", "really", "quite",
}
# Brand/product tokens excluded from language classification
BRAND_TOKENS = {"shin", "ramyun", "ramyum", "nongshim", "nongshim's", "maggi",
                "korea", "korean", "malaysia", "malaysian"}

for w in KNOWN_MALAY:
    if w not in word_lang:
        word_lang[w] = "Malay"
for w in KNOWN_ENGLISH:
    if w not in word_lang:
        word_lang[w] = "English"

def get_token_language(token):
    """Return the language label for a token, or 'ambiguous' if unknown."""
    t = token.casefold()
    if t in BRAND_TOKENS:
        return "brand"
    return word_lang.get(t, "ambiguous")

print(f"Word-language lookup entries: {len(word_lang)}")
print(f"Known Malay content words: {len(KNOWN_MALAY)}")
print(f"Known English content words: {len(KNOWN_ENGLISH)}")

Word-language lookup entries: 217
Known Malay content words: 63
Known English content words: 52


## 4. Generate n-grams from unfiltered sentence tokens

N-grams are extracted within sentence boundaries to avoid spurious cross-sentence pairs. We use `tokens_before` (unfiltered) so that function words and code-switching markers are preserved. Stop-word-filtered n-grams are computed separately for content-word patterns.

In [4]:
def extract_ngrams(tokens, n):
    """Extract n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def corpus_ngrams(reviews, n, use_field="sentence_tokens_before", filter_stopwords=False):
    """Extract n-grams within sentence boundaries across the full corpus.
    
    Returns Counter of n-gram tuples and a dict mapping n-gram -> set of (origin, review_id).
    """
    counts = Counter()
    review_sets = {}  # ngram -> set of (origin, review_id)
    for row in reviews:
        if row.get("included") is False:
            continue
        key = (row["origin"], row["review_id"])
        for sentence in row.get(use_field, []):
            tokens = [t.casefold() for t in sentence]
            if filter_stopwords:
                tokens = [t for t in tokens if t not in stopword_set]
            grams = extract_ngrams(tokens, n)
            for gram in grams:
                counts[gram] += 1
                review_sets.setdefault(gram, set()).add(key)
    return counts, review_sets

# Unfiltered bigrams and trigrams (within sentences)
bigram_counts, bigram_reviews = corpus_ngrams(reviews, 2)
trigram_counts, trigram_reviews = corpus_ngrams(reviews, 3)

# Filtered bigrams and trigrams (stop words removed)
bigram_filtered, bigram_filt_reviews = corpus_ngrams(reviews, 2, filter_stopwords=True)
trigram_filtered, trigram_filt_reviews = corpus_ngrams(reviews, 3, filter_stopwords=True)

total_unfiltered = sum(len(row["tokens_before"]) for row in reviews if row.get("included") is not False)

print(f"Unfiltered bigram types: {len(bigram_counts):,}")
print(f"Unfiltered trigram types: {len(trigram_counts):,}")
print(f"Filtered bigram types (stop words removed): {len(bigram_filtered):,}")
print(f"Filtered trigram types (stop words removed): {len(trigram_filtered):,}")
print(f"Total unfiltered tokens: {total_unfiltered:,}")

Unfiltered bigram types: 1,666
Unfiltered trigram types: 1,761
Filtered bigram types (stop words removed): 1,259
Filtered trigram types (stop words removed): 1,185
Total unfiltered tokens: 2,252


## 5. Most frequent bigrams

The tables below show the top 25 bigrams by frequency, both before and after stop-word filtering. Unfiltered bigrams reveal grammatical and structural patterns, while filtered bigrams highlight content-word co-occurrences related to product attributes.

In [5]:
def ngram_to_df(counts, review_sets, total_tokens, top_n=25, label="bigram"):
    """Convert n-gram counts to a DataFrame with frequency and review coverage."""
    rows = []
    for gram, count in counts.most_common(top_n):
        gram_str = " ".join(gram)
        langs = [get_token_language(t) for t in gram]
        lang_pair = " + ".join(langs)
        # Determine if cross-language
        real_langs = {l for l in langs if l not in ("brand", "ambiguous")}
        is_cross = len(real_langs) > 1
        rows.append({
            "ngram": gram_str,
            "frequency": count,
            "reviews_containing": len(review_sets.get(gram, set())),
            "per_1000_tokens": round(count / total_tokens * 1000, 3) if total_tokens else 0,
            "token_languages": lang_pair,
            "cross_language": is_cross,
        })
    return pd.DataFrame(rows)

print("=" * 70)
print("TOP 25 UNFILTERED BIGRAMS")
print("=" * 70)
bigram_df = ngram_to_df(bigram_counts, bigram_reviews, total_unfiltered, 25)
display(bigram_df)

print("\n" + "=" * 70)
print("TOP 25 FILTERED BIGRAMS (stop words removed)")
print("=" * 70)
bigram_filt_df = ngram_to_df(bigram_filtered, bigram_filt_reviews, total_unfiltered, 25)
display(bigram_filt_df)

TOP 25 UNFILTERED BIGRAMS
              ngram  frequency  ...      token_languages  cross_language
0       shin ramyun         57  ...        brand + brand           False
1       the noodles         17  ...    English + English           False
2           and the         11  ...    English + English           False
3           but the          9  ...  ambiguous + English           False
4       noodles are          7  ...    English + English           False
5   instant noodles          7  ...    English + English           False
6         saya suka          7  ...        Malay + Malay           False
7       the flavour          6  ...    English + English           False
8            of the          6  ...    English + English           False
9        kenyal dan          6  ...        Malay + Malay           False
10        bagi saya          6  ...    ambiguous + Malay           False
11          mi shin          6  ...        Malay + brand           False
12         the soup      

## 6. Most frequent trigrams

Trigrams capture longer patterns such as attribute-evaluation phrases (e.g., 'kuah sedap sangat', 'not too spicy') and three-word code-switching sequences.

In [6]:
print("=" * 70)
print("TOP 25 UNFILTERED TRIGRAMS")
print("=" * 70)
trigram_df = ngram_to_df(trigram_counts, trigram_reviews, total_unfiltered, 25, "trigram")
display(trigram_df)

print("\n" + "=" * 70)
print("TOP 25 FILTERED TRIGRAMS (stop words removed)")
print("=" * 70)
trigram_filt_df = ngram_to_df(trigram_filtered, trigram_filt_reviews, total_unfiltered, 25, "trigram")
display(trigram_filt_df)

TOP 25 UNFILTERED TRIGRAMS
                   ngram  ...  cross_language
0         mi shin ramyun  ...           False
1      shin ramyun punya  ...           False
2     kenyal dan kuahnya  ...           False
3        the noodles are  ...           False
4         shin ramyun ni  ...           False
5       kuah shin ramyun  ...           False
6         shin ramyun is  ...           False
7         of the noodles  ...           False
8            i love this  ...           False
9      shin ramyun sedap  ...           False
10        the flavour of  ...           False
11           be a little  ...           False
12  my personal favorite  ...           False
13          and the soup  ...           False
14        for people who  ...           False
15            i like the  ...           False
16        texture of the  ...           False
17         not too spicy  ...           False
18         too spicy but  ...           False
19           the soup is  ...           False
20     

## 7. Cross-language bigrams: code-switching at the word boundary

Cross-language bigrams are pairs where one token is classified as English and the other as Malay (or vice versa). These bigrams directly represent **intra-sentential code-switching at word boundaries** and are key evidence for understanding how reviewers mix languages when discussing product attributes.

Brand/product tokens and ambiguous words are excluded from cross-language classification to avoid false positives.

In [7]:
def get_cross_language_ngrams(counts, review_sets, total_tokens, top_n=30):
    """Extract n-grams where adjacent tokens are from different identified languages."""
    cross = []
    for gram, count in counts.most_common():
        langs = [get_token_language(t) for t in gram]
        real_langs = [l for l in langs if l not in ("brand", "ambiguous")]
        if len(real_langs) >= 2 and len(set(real_langs)) > 1:
            gram_str = " ".join(gram)
            lang_pair = " → ".join(langs)
            cross.append({
                "ngram": gram_str,
                "frequency": count,
                "reviews_containing": len(review_sets.get(gram, set())),
                "per_1000_tokens": round(count / total_tokens * 1000, 3) if total_tokens else 0,
                "language_direction": lang_pair,
            })
        if len(cross) >= top_n:
            break
    return pd.DataFrame(cross)

print("=" * 70)
print("TOP CROSS-LANGUAGE BIGRAMS (English ↔ Malay word boundaries)")
print("=" * 70)
cross_bigrams_df = get_cross_language_ngrams(bigram_counts, bigram_reviews, total_unfiltered, 30)
if len(cross_bigrams_df) > 0:
    display(cross_bigrams_df)
    print(f"\nTotal cross-language bigram types: {len(cross_bigrams_df)}")
else:
    print("No cross-language bigrams found with current classifications.")

print("\n" + "=" * 70)
print("TOP CROSS-LANGUAGE TRIGRAMS")
print("=" * 70)
cross_trigrams_df = get_cross_language_ngrams(trigram_counts, trigram_reviews, total_unfiltered, 20)
if len(cross_trigrams_df) > 0:
    display(cross_trigrams_df)
else:
    print("No cross-language trigrams found with current classifications.")

TOP CROSS-LANGUAGE BIGRAMS (English ↔ Malay word boundaries)
             ngram  frequency  ...  per_1000_tokens  language_direction
0        rasa kick          1  ...            0.444     Malay → English
1         kick dgn          1  ...            0.444     English → Malay
2        dgn spice          1  ...            0.444     Malay → English
3     kari instant          1  ...            0.444     Malay → English
4     noodle dalam          1  ...            0.444     English → Malay
5          yg soup          1  ...            0.444     Malay → English
6         soup mmg          1  ...            0.444     English → Malay
7     flavour yang          1  ...            0.444     English → Malay
8         kari has          1  ...            0.444     Malay → English
9        kuah rich          1  ...            0.444     Malay → English
10        rich dan          1  ...            0.444     English → Malay
11       memang my          1  ...            0.444     Malay → English
12 

## 8. N-gram patterns by review language label

Compare the most frequent content-word bigrams across English, Malay and code-switched reviews. These are inherited language labels from the source Sheet, not automatic detection. The comparison shows which attribute-related word pairs appear in each language group.

In [8]:
def ngrams_by_language(reviews, n, top_n=15):
    """Extract top filtered n-grams grouped by review language label."""
    lang_groups = {}
    for row in reviews:
        if row.get("included") is False:
            continue
        lang = row.get("language", "unknown")
        if lang not in lang_groups:
            lang_groups[lang] = Counter()
        for sentence in row.get("sentence_tokens_before", []):
            tokens = [t.casefold() for t in sentence if t.casefold() not in stopword_set]
            for gram in extract_ngrams(tokens, n):
                lang_groups[lang][gram] += 1
    
    results = {}
    for lang in sorted(lang_groups):
        rows = []
        for gram, count in lang_groups[lang].most_common(top_n):
            gram_str = " ".join(gram)
            token_langs = [get_token_language(t) for t in gram]
            rows.append({
                "ngram": gram_str,
                "frequency": count,
                "token_languages": " + ".join(token_langs),
            })
        results[lang] = pd.DataFrame(rows)
    return results

bigrams_by_lang = ngrams_by_language(reviews, 2, 15)

for lang, df in bigrams_by_lang.items():
    print(f"\n{'=' * 50}")
    print(f"TOP 15 FILTERED BIGRAMS IN {lang.upper()} REVIEWS")
    print(f"{'=' * 50}")
    display(df)


TOP 15 FILTERED BIGRAMS IN CODE-SWITCHED REVIEWS
            ngram  frequency        token_languages
0     shin ramyun         21          brand + brand
1    ramyun punya          5          brand + Malay
2      maggi kari          3          brand + Malay
3         mi shin          3          Malay + brand
4     sedap jugak          2          Malay + Malay
5     makan malam          2      Malay + ambiguous
6       tak perlu          2          Malay + Malay
7       kuah kari          2          Malay + Malay
8      bila lapar          2      Malay + ambiguous
9      asam laksa          2  ambiguous + ambiguous
10      kuah shin          2          Malay + brand
11  ramyun memang          2          brand + Malay
12      pedas but          2      Malay + ambiguous
13   ramyun sedap          2          brand + Malay
14    ramyun agak          2          brand + Malay

TOP 15 FILTERED BIGRAMS IN ENGLISH REVIEWS
                ngram  frequency        token_languages
0         shin ram

## 9. Attribute-focused n-gram analysis

Group n-grams by the product attribute they discuss. This directly addresses the research question about which attributes are frequently discussed. The attribute grouping is based on the presence of known attribute terms.

In [9]:
# Define attribute categories based on the assignment's scope
ATTRIBUTE_TERMS = {
    "taste": {"taste", "flavour", "flavor", "rasa", "sedap", "enak", "lazat", "nikmat",
              "spicy", "pedas", "salty", "masin", "bland", "delicious"},
    "texture": {"texture", "springy", "chewy", "kenyal", "lembik", "firm", "soft",
                "soggy", "crispy", "rangup", "thick", "thin", "thinner"},
    "broth/soup": {"soup", "broth", "kuah", "kuahnya", "kari", "cair", "pekat", "rich"},
    "price/value": {"price", "harga", "mahal", "pricey", "worth", "value", "cheap", "affordable"},
    "packaging": {"packaging", "bungkusan", "packet", "portion", "sachet"},
    "preparation": {"cook", "cooking", "masak", "rebus", "goreng", "boil"},
}

def attribute_ngrams(counts, review_sets, total_tokens, n_gram_size=2):
    """Classify n-grams by which attribute category they relate to."""
    results = {attr: [] for attr in ATTRIBUTE_TERMS}
    for gram, count in counts.most_common():
        gram_tokens = set(gram)
        for attr, terms in ATTRIBUTE_TERMS.items():
            if gram_tokens & terms:
                langs = [get_token_language(t) for t in gram]
                real_langs = {l for l in langs if l not in ("brand", "ambiguous")}
                results[attr].append({
                    "ngram": " ".join(gram),
                    "frequency": count,
                    "reviews": len(review_sets.get(gram, set())),
                    "token_languages": " + ".join(langs),
                    "cross_language": len(real_langs) > 1,
                })
    return results

attr_bigrams = attribute_ngrams(bigram_filtered, bigram_filt_reviews, total_unfiltered)

print("ATTRIBUTE-RELATED BIGRAMS (filtered, sorted by frequency)")
print("=" * 70)
for attr in ATTRIBUTE_TERMS:
    items = attr_bigrams[attr][:10]  # top 10 per attribute
    if items:
        print(f"\n--- {attr.upper()} ---")
        df = pd.DataFrame(items)
        display(df)
    else:
        print(f"\n--- {attr.upper()} --- (no bigrams found)")

ATTRIBUTE-RELATED BIGRAMS (filtered, sorted by frequency)

--- TASTE ---
           ngram  frequency  reviews      token_languages  cross_language
0   ramyun sedap          4        4        brand + Malay           False
1      too spicy          3        3  ambiguous + English           False
2  terlalu masin          3        3    ambiguous + Malay           False
3    sedap jugak          2        2        Malay + Malay           False
4      spicy but          2        2  English + ambiguous           False
5   sangat sedap          2        2        Malay + Malay           False
6  terlalu pedas          2        2    ambiguous + Malay           False
7   sedap sangat          2        2        Malay + Malay           False
8   flavour okay          2        2  English + ambiguous           False
9      pedas but          2        2    Malay + ambiguous           False

--- TEXTURE ---
             ngram  frequency  reviews        token_languages  cross_language
0   kenyal kuahnya

## 10. Summary statistics and save outputs

Save all n-gram tables for the report and further analysis.

In [10]:
import hashlib

# Save full unfiltered bigram frequencies (top 50)
full_bigram_df = ngram_to_df(bigram_counts, bigram_reviews, total_unfiltered, 50)
full_bigram_df.to_csv(OUTPUT_DIR / "bigram_frequency_unfiltered.csv", index=False, encoding="utf-8-sig")

# Save filtered bigram frequencies (top 50)
full_bigram_filt_df = ngram_to_df(bigram_filtered, bigram_filt_reviews, total_unfiltered, 50)
full_bigram_filt_df.to_csv(OUTPUT_DIR / "bigram_frequency_filtered.csv", index=False, encoding="utf-8-sig")

# Save full unfiltered trigram frequencies (top 50)
full_trigram_df = ngram_to_df(trigram_counts, trigram_reviews, total_unfiltered, 50, "trigram")
full_trigram_df.to_csv(OUTPUT_DIR / "trigram_frequency_unfiltered.csv", index=False, encoding="utf-8-sig")

# Save filtered trigram frequencies (top 50)
full_trigram_filt_df = ngram_to_df(trigram_filtered, trigram_filt_reviews, total_unfiltered, 50, "trigram")
full_trigram_filt_df.to_csv(OUTPUT_DIR / "trigram_frequency_filtered.csv", index=False, encoding="utf-8-sig")

# Save cross-language bigrams
all_cross_bigrams = get_cross_language_ngrams(bigram_counts, bigram_reviews, total_unfiltered, 100)
all_cross_bigrams.to_csv(OUTPUT_DIR / "cross_language_bigrams.csv", index=False, encoding="utf-8-sig")

# Save attribute-grouped bigrams
attr_rows = []
for attr, items in attr_bigrams.items():
    for item in items[:15]:  # top 15 per attribute
        item["attribute"] = attr
        attr_rows.append(item)
pd.DataFrame(attr_rows).to_csv(OUTPUT_DIR / "attribute_bigrams.csv", index=False, encoding="utf-8-sig")

# Save by-language bigrams
lang_rows = []
for lang, df in bigrams_by_lang.items():
    for _, row in df.iterrows():
        lang_rows.append({"review_language": lang, **row.to_dict()})
pd.DataFrame(lang_rows).to_csv(OUTPUT_DIR / "bigrams_by_language.csv", index=False, encoding="utf-8-sig")

# Build and save run manifest
inputs = [PROCESSED_DIR / f"{origin}_preprocessed.jsonl" for origin in ("real", "ai")]
inputs.append(PROCESSED_DIR / "stopwords_used.csv")
manifest = {
    "review_count": len(reviews),
    "unfiltered_tokens": total_unfiltered,
    "unfiltered_bigram_types": len(bigram_counts),
    "unfiltered_trigram_types": len(trigram_counts),
    "filtered_bigram_types": len(bigram_filtered),
    "filtered_trigram_types": len(trigram_filtered),
    "cross_language_bigram_types": len(all_cross_bigrams),
    "attribute_categories": list(ATTRIBUTE_TERMS.keys()),
    "sentence_bounded": True,
    "input_sha256": {
        str(p.relative_to(PROJECT_DIR)).replace("\\", "/"): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in inputs
    },
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

print("\n" + "=" * 70)
print("N-GRAM ANALYSIS SUMMARY")
print("=" * 70)
print(f"Reviews analysed: {len(reviews)}")
print(f"Unfiltered tokens: {total_unfiltered:,}")
print(f"Unfiltered bigram types: {len(bigram_counts):,}")
print(f"Unfiltered trigram types: {len(trigram_counts):,}")
print(f"Filtered bigram types: {len(bigram_filtered):,}")
print(f"Filtered trigram types: {len(trigram_filtered):,}")
print(f"Cross-language bigram types: {len(all_cross_bigrams)}")
print(f"\nSaved files in {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")


N-GRAM ANALYSIS SUMMARY
Reviews analysed: 125
Unfiltered tokens: 2,252
Unfiltered bigram types: 1,666
Unfiltered trigram types: 1,761
Filtered bigram types: 1,259
Filtered trigram types: 1,185
Cross-language bigram types: 28

Saved files in C:\Users\Admin\OneDrive\Documents\GitHub\CPC353-Assignment-1\data\analysis\ngram:
  attribute_bigrams.csv
  bigram_frequency_filtered.csv
  bigram_frequency_unfiltered.csv
  bigrams_by_language.csv
  cross_language_bigrams.csv
  run_manifest.json
  trigram_frequency_filtered.csv
  trigram_frequency_unfiltered.csv


## 11. Interpretation and limits

The n-gram analysis complements the concordance and stop-word analyses:

- **Unfiltered bigrams** show structural and grammatical patterns, including function-word combinations.
- **Filtered bigrams** highlight content-word co-occurrences related to product attributes (taste, texture, broth, price, packaging).
- **Cross-language bigrams** identify word boundaries where reviewers switch between English and Malay, providing quantitative evidence for intra-sentential code-switching.
- **Language-grouped bigrams** compare attribute-discussion patterns across English, Malay, and code-switched reviews.

**Limitations:**
- Token-level language labels are from a project-authored lookup, not automatic language identification. Ambiguous tokens are excluded from cross-language counts.
- N-grams are extracted within estimated sentence boundaries; cross-sentence patterns are not captured.
- Brand and product-name tokens are excluded from language classification but remain in the n-grams.
- The corpus is a convenience sample with both collected and AI-generated reviews; n-gram frequencies do not represent category-wide patterns.
- Frequency alone does not establish meaning or sentiment. N-grams must be interpreted alongside the concordance evidence and original reviews.

**Reference:** C. D. Manning and H. Schütze, *Foundations of Statistical Natural Language Processing*, MIT Press, 1999, ch. 6, "Statistical Inference: n-gram Models over Sparse Data."